# 1.0 TOKENIZE TEXT

In [1]:
with open("../the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of charachter:", len(raw_text))
print(raw_text[:99])

Total number of charachter: 20480
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [2]:
import re 
text = "Hello world, This is a text."
result = re.split(r"(\s)", text)
print(result)

['Hello', ' ', 'world,', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'text.']


In [3]:
result = re.split(r"([,.]|\s)", text)
print(result)

['Hello', ' ', 'world', ',', '', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'text', '.', '']


In [4]:
result = [item for item in result if item.strip()]
print(result)

['Hello', 'world', ',', 'This', 'is', 'a', 'text', '.']


In [5]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'Is', ' ', 'this', '--', '', ' ', 'a', ' ', 'test', '?', '']


In [6]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


In [7]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


# 1.1 Converting Tokens into token IDS

In [8]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [9]:
vocab = {token: integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()): 
    print(item)
    if i>50: 
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)
('His', 51)


In [10]:
vocab["I"]

53

In [11]:
class SimpleTokenizerV1: 
    def __init__(self, vocab): 
        """
        params:
            vocab: Dict = as {string: id}
        """
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text): 
        """ This turns string into id."""
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids): 
        """ This decode the ids into strings"""

        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text


In [12]:
tokenizer = SimpleTokenizerV1(vocab)
text = "I"
ids = tokenizer.encode(text)
print(ids)

[53]


Because the `Hi` word is not in the vocabulary since the vocabulary only contains the words in the-verdict.txt

## 1.1.1 Adding special context tokens

In [13]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}
print(len(vocab.items()))

1132


In [14]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [15]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int
    #A
        else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        #B
        return text

In [16]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [17]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))


[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [18]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [19]:
import tiktoken


In [20]:
tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

In [21]:
integers

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 554,
 262,
 4252,
 18250,
 8812,
 2114,
 286,
 617,
 34680,
 27271,
 13]

In [22]:
text = tokenizer.decode(integers)

In [23]:
text

'Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.'

In [24]:
text_2 = "Akwirw ier came to me and said hi!"
integers = tokenizer.encode(text_2)

In [25]:
integers

[33901, 86, 343, 86, 220, 959, 1625, 284, 502, 290, 531, 23105, 0]

In [26]:
text_2_decoded = tokenizer.decode(integers)
text_2_decoded

'Akwirw ier came to me and said hi!'

# 1.2 Data Sampling with a sliding window

In [27]:
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5146


In [28]:
enc_sample = enc_text[50:]

In [29]:
enc_sample

[290,
 4920,
 2241,
 287,
 257,
 4489,
 64,
 319,
 262,
 34686,
 41976,
 13,
 357,
 10915,
 314,
 2138,
 1807,
 340,
 561,
 423,
 587,
 10598,
 393,
 28537,
 2014,
 198,
 198,
 1,
 464,
 6001,
 286,
 465,
 13476,
 1,
 438,
 5562,
 373,
 644,
 262,
 1466,
 1444,
 340,
 13,
 314,
 460,
 3285,
 9074,
 13,
 46606,
 536,
 5469,
 438,
 14363,
 938,
 4842,
 1650,
 353,
 438,
 2934,
 489,
 3255,
 465,
 48422,
 540,
 450,
 67,
 3299,
 13,
 366,
 5189,
 1781,
 340,
 338,
 1016,
 284,
 3758,
 262,
 1988,
 286,
 616,
 4286,
 705,
 1014,
 510,
 26,
 475,
 314,
 836,
 470,
 892,
 286,
 326,
 11,
 1770,
 13,
 8759,
 2763,
 438,
 1169,
 2994,
 284,
 943,
 17034,
 318,
 477,
 314,
 892,
 286,
 526,
 383,
 1573,
 11,
 319,
 9074,
 13,
 536,
 5469,
 338,
 11914,
 11,
 33096,
 663,
 4808,
 3808,
 62,
 355,
 996,
 484,
 547,
 12548,
 287,
 281,
 13079,
 410,
 12523,
 286,
 22353,
 13,
 843,
 340,
 373,
 407,
 691,
 262,
 9074,
 13,
 536,
 48819,
 508,
 25722,
 276,
 13,
 11161,
 407,
 262,
 40123,
 18113,


In [30]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"Input X: {x}")
print(f"Output Y:   {y}")

Input X: [290, 4920, 2241, 287]
Output Y:   [4920, 2241, 287, 257]


In [31]:
for i in range(1, context_size+1): 
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(f"{context} ----> {desired}")

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [32]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [33]:
import torch 
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride): 
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride): 
            input_chunk = token_ids[i: i + max_length]
            target_chunk = token_ids[i+1: i+max_length+1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self): 
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]



In [34]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0): 
    tokenizer = tiktoken.get_encoding('gpt2')
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    dataloader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [44]:
with open("../the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
raw_text, batch_size=1, max_length=4, stride=2, shuffle=False)
data_iter = iter(dataloader)
#A
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [45]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[2885, 1464, 1807, 3619]]), tensor([[1464, 1807, 3619,  402]])]


In [46]:
dataloader2 = create_dataloader_v1(
    raw_text, batch_size=1, max_length=2, stride=2, shuffle=False
)

data_iter = iter(dataloader2)
#A
first_batch = next(data_iter)
print(first_batch)

[tensor([[ 40, 367]]), tensor([[ 367, 2885]])]


In [47]:
dataloader3 = create_dataloader_v1(
    raw_text, batch_size=1, max_length=8, stride=2, shuffle=False
)

data_iter = iter(dataloader3)
#A
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]


In [50]:
dataloader4 = create_dataloader_v1(
    raw_text, batch_size=4, max_length=256, stride=128, shuffle=False
)

data_iter = iter(dataloader4)
#A
first_batch = next(data_iter)
print(first_batch)

[tensor([[   40,   367,  2885,  ...,   198,  5779, 28112],
        [  286,   616,  4286,  ...,   470,   910,   416],
        [10197,   832,   262,  ...,  9074,    13,   402],
        [ 4150,     8,  3688,  ..., 14093,   656,   465]]), tensor([[  367,  2885,  1464,  ...,  5779, 28112, 10197],
        [  616,  4286,   705,  ...,   910,   416,  4150],
        [  832,   262, 46475,  ...,    13,   402,   271],
        [    8,  3688,   284,  ...,   656,   465,  1021]])]


In [51]:
len(dataloader), len(dataloader2), len(dataloader3), len(dataloader4)

(2571, 2572, 2569, 9)

In [52]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   13,   198,   198,     1],
        [  290,   673,  1297,  8276],
        [   11, 17728,   257,  8500],
        [  765,   683,   284,  2883],
        [  438, 13893,   339,  1422],
        [   13,   402,   271, 10899],
        [32073,    13,   198,   198],
        [ 5287,   852, 37492,   416]])

Targets:
 tensor([[  198,   198,     1,  5812],
        [  673,  1297,  8276,  2073],
        [17728,   257,  8500,  4417],
        [  683,   284,  2883,  2241],
        [13893,   339,  1422,   470],
        [  402,   271, 10899,    11],
        [   13,   198,   198,     1],
        [  852, 37492,   416,   262]])


# 1.3 Creating token embeddings

In [55]:
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3

torch.manual_seed(123)

embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [56]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [57]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [58]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [61]:
token_embedding_layer.weight.shape

torch.Size([50257, 256])

In [65]:
token_embedding_layer.weight[1]

tensor([ 1.2277, -0.4297, -2.2121, -0.3780,  0.9838, -1.0895, -0.6378, -0.6031,
        -1.7753, -0.7490,  0.2781, -0.9621, -0.4223, -1.1036,  0.8398, -1.0029,
        -0.2835, -0.3767, -0.0306, -0.0894, -0.1965, -0.9713,  0.2790, -0.7587,
         1.0669, -0.2985,  0.8558,  1.6098, -1.1893,  1.1677,  0.6220,  2.5737,
        -1.6179,  0.2265, -0.4382,  0.3265, -1.5786, -1.3995,  0.2425,  0.3648,
        -1.1753,  1.7825,  1.7524, -0.2135,  0.4095,  0.0465,  0.5468,  1.1478,
        -0.8614,  0.5338,  0.9376, -0.9225,  0.7047, -0.2722,  0.0781, -0.1134,
         2.3902, -1.4256, -0.4619, -1.5539, -0.3338,  0.2405, -0.0334,  1.5544,
        -0.2936, -1.8027, -0.6933,  1.7409,  0.2698,  0.9595,  0.7744,  1.8721,
         1.0264, -0.5670, -0.2658, -1.1116, -1.3696, -0.6534, -2.1587,  0.8093,
         1.8388, -0.9473,  0.1419,  0.3696, -0.0174, -0.9575,  1.2968,  0.6833,
         0.4343, -0.1340, -2.1467, -1.7984, -0.6822, -0.5191,  0.1669, -1.2620,
        -0.2443,  0.1327,  1.0875, -0.10

In [ ]:
max_length = 4
dataloader = create_dataloader_v1(
raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [67]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [68]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [72]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


In [78]:
input_embeddings

tensor([[[ 2.2288,  0.5619,  0.8286,  ..., -0.6272, -0.2987,  0.8900],
         [ 2.0903, -0.4664, -0.0593,  ...,  0.9115, -1.0493, -1.6473],
         [-0.7158, -0.8304,  1.2494,  ...,  2.3952,  1.8773,  0.8051],
         [ 0.2703,  0.4029,  3.0514,  ...,  0.3595, -1.4548,  0.8310]],

        [[ 3.2835,  1.1749, -1.4150,  ..., -0.3281,  2.4332,  0.6924],
         [-0.2199, -0.9114, -0.1750,  ...,  1.5337, -0.1998,  0.1462],
         [ 1.5197, -1.4240,  0.4391,  ...,  1.0494, -1.4318,  2.3057],
         [ 0.2893,  0.8346, -0.1884,  ...,  1.9602,  0.8709,  0.8796]],

        [[ 0.9662,  0.0952, -0.4640,  ..., -1.0320,  1.6290,  1.7771],
         [ 2.4468, -0.2154,  1.4984,  ...,  1.8766,  0.5595, -0.1423],
         [-0.3856, -2.5393,  1.1556,  ...,  3.6157,  1.3267,  0.4944],
         [-0.2487, -0.5275,  2.0009,  ...,  0.2930,  0.5977,  1.3300]],

        ...,

        [[ 0.1219,  0.3991, -3.2740,  ..., -1.1921,  2.6637,  2.6728],
         [ 1.2438, -1.6436, -1.1101,  ..., -0.7464, -0.98

In [79]:
torch.save(input_embeddings, "input_embeddings.pkl")

# Summary

1-) Input Text<br>
2-) Tokenized Text <br>
3-) Token IDs <br>
4-) Token Embeddings + Positional Embeddings <br>
5-) Input Embeddings is ready ^^^